# Experiment 5 - Cost of Noise Flooding

CKKS decryption returns the message plus an error term that depends on the secret key, so publishing decrypted values leaks information about that key. OpenFHE mitigates this by adding noise before decryption.

This experiment measures what that mode costs, and checks whether each library's decryption is randomised.

Beyond the scope of the project proposal; included as an additional experiment and not part of the report's main results.

## 1. Install and load the project

In [ ]:
!pip install -q tenseal openfhe numpy matplotlib

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/To2004/confidential-computing-project.git"

# Works both on a fresh Colab runtime and inside a local checkout.
if not os.path.exists("src/benchmark_harness.py"):
    if not os.path.exists("confidential-computing-project"):
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("confidential-computing-project")

# Absolute, so imports survive a later change of directory, and guarded so
# re-running this cell does not stack duplicate entries.
SRC = os.path.abspath("src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Write this notebook's output to its own directory, so running it does not
# overwrite the results and figures the report was built from.
import benchmark_harness as harness
import project_paths

os.makedirs("notebook_output", exist_ok=True)
project_paths.RESULTS_DIR = "notebook_output"
project_paths.FIGURES_DIR = "notebook_output"

print("working directory:", os.getcwd())
print("report uses", harness.DEFAULT_REPEATS, "repetitions per measurement")

## 2. Check which libraries loaded

The OpenFHE wheel ships a binary built for CPython 3.8. On a newer Python it
installs but fails to import.

This experiment needs OpenFHE and will not run without it.

In [ ]:
import importlib

for name in ("tenseal", "openfhe"):
    try:
        importlib.import_module(name)
        print(f"{name}: available")
    except Exception as exc:
        print(f"{name}: not available -- {exc}")

# This experiment measures OpenFHE itself, so stop here with a clear message
# rather than letting the benchmark below fail inside a subprocess.
try:
    import openfhe
except ImportError:
    raise RuntimeError(
        "This notebook needs OpenFHE. Its wheel is built for CPython 3.8, so it "
        "does not import on a newer runtime such as Colab. Run this notebook in "
        "the he38 environment described in the project README."
    ) from None

## 3. Run the experiment

`REPEATS` is the number of timed repetitions per measurement. The report uses 1000 on a reserved compute node; this notebook uses fewer so it finishes in a few minutes. The numbers it prints will therefore be noisier than the ones in the report.

A smaller cohort is used here because the mode requires running the circuit twice.

In [ ]:
REPEATS = 20

!python src/ind_cpad_flooding.py --repeats {REPEATS} --warmup 5 --patients 64 --output notebook_output/ind_cpad_results.json

## 4. Results

In [ ]:
import plot_results as pr
from IPython.display import Image, display

pr.apply_style()
pr.chart_ind_cpad(pr.load("ind_cpad_results.json"))
display(Image("notebook_output/chart_ind_cpad.png"))